<a href="https://colab.research.google.com/github/shreya1111/flyrank-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreya1111/flyrank-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Classification.** The task is to predict whether a customer will 'churn' (cancel their service or stop using a product) within a certain future time window. This is a binary outcome (churn/no churn), which is a classic classification problem. We are assigning a discrete label to each customer.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Example: Simulate some churn data for understanding.
import pandas as pd
import numpy as np

# Dummy data to illustrate classification target
data = {
    'customer_id': range(1, 11),
    'feature_1': np.random.rand(10),
    'feature_2': np.random.randint(0, 100, 10),
    'churn': [0, 1, 0, 0, 1, 0, 0, 1, 0, 0] # 0 = no churn, 1 = churn
}
df_classification_example = pd.DataFrame(data)
print("Example of a classification target:")
print(df_classification_example[['customer_id', 'churn']].head())

Example of a classification target:
   customer_id  churn
0            1      0
1            2      1
2            3      0
3            4      0
4            5      1


## 2. Target or proxy

**Target: Customer Churn.** The target would be a binary variable indicating whether a customer has churned or not. This label comes from an **observed outcome**: we define churn based on actual customer behavior, such as a subscription cancellation event, account inactivity for a certain period, or lack of purchases within a defined timeframe (e.g., 90 days).

For instance, if a customer's subscription ends and is not renewed within 7 days, they are labeled 'churned'. If it's still active, they are 'not churned'.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Example: Define the churn rule (conceptual code).

def define_churn_label(customer_data):
    # Simulate a check for subscription end and non-renewal
    # In a real scenario, this would query a database for actual events.
    if customer_data['subscription_status'] == 'ended' and customer_data['days_since_renewal_due'] > 7:
        return 1  # Churned
    else:
        return 0  # Not churned

# Example usage with dummy customer data
dummy_customer1 = {'subscription_status': 'active', 'days_since_renewal_due': 0}
dummy_customer2 = {'subscription_status': 'ended', 'days_since_renewal_due': 10}

print(f"Customer 1 churn status: {define_churn_label(dummy_customer1)}")
print(f"Customer 2 churn status: {define_churn_label(dummy_customer2)}")

Customer 1 churn status: 0
Customer 2 churn status: 1


## 3. Success metric

**F1-Score.** For churn prediction, we want to balance both false positives (predicting churn when a customer won't) and false negatives (failing to predict churn when they will). A false negative can be costly as we miss an opportunity for intervention, while too many false positives can lead to wasted resources on customers who were never going to churn.

The F1-Score is the harmonic mean of precision and recall, providing a single metric that punishes extreme values of either. It's particularly suitable when the classes are imbalanced (e.g., many more non-churners than churners), as is often the case with churn data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Example: Illustrate F1-score with dummy predictions.
from sklearn.metrics import f1_score, precision_score, recall_score

# True labels (actual churn status)
y_true = np.array([0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1])
# Predicted labels (from a hypothetical model)
y_pred = np.array([0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1]) # One false negative, one false positive

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"True Labels:    {y_true}")
print(f"Predicted Labels: {y_pred}\n")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1-Score:  {f1:.2f}")

True Labels:    [0 1 0 0 1 0 0 1 0 0 0 1 0 0 1]
Predicted Labels: [0 1 0 0 0 0 0 1 0 0 0 1 1 0 1]

Precision: 0.80
Recall:    0.80
F1-Score:  0.80


## 4. The unit of analysis, as a real dataframe

The unit of analysis is a **customer** at a specific point in time (e.g., end of quarter). Each row in our dataframe represents one customer, with various features (e.g., usage data, demographic information, contract details) and the target churn label for a future period. This snapshot approach allows us to predict churn based on past and current customer state.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Create a dummy DataFrame representing the unit of analysis (customer).
import pandas as pd
import numpy as np

np.random.seed(42)
num_customers = 100

data = {
    'customer_id': range(1, num_customers + 1),
    'age': np.random.randint(18, 70, num_customers),
    'monthly_spend': np.round(np.random.normal(50, 20, num_customers), 2),
    'contract_type': np.random.choice(['month-to-month', 'one-year', 'two-year'], num_customers),
    'total_data_usage_gb': np.round(np.random.normal(100, 50, num_customers), 2),
    'customer_service_calls': np.random.randint(0, 5, num_customers),
    'is_senior_citizen': np.random.choice([0, 1], num_customers, p=[0.8, 0.2]),
    'churn_next_quarter': np.random.choice([0, 1], num_customers, p=[0.85, 0.15]) # 15% churn rate
}

df_customers = pd.DataFrame(data)

print("Unit of Analysis: Each row represents one customer.")
display(df_customers.head())

Unit of Analysis: Each row represents one customer.


,customer_id,age,monthly_spend,contract_type,total_data_usage_gb,customer_service_calls,is_senior_citizen,churn_next_quarter
0,1,56,12.49,one-year,55.81,4,0,1
1,2,69,22.66,two-year,107.69,2,0,0
2,3,46,62.73,two-year,102.91,4,0,0
3,4,32,31.87,one-year,42.85,3,0,0
4,5,60,59.52,month-to-month,117.89,0,0,0


## 5. Why ML beats a fixed rule here

Customer churn is a complex phenomenon influenced by a multitude of interacting factors that are difficult to capture with simple `if-else` statements. For example, a customer's likelihood to churn isn't just about their monthly spend, but rather a combination of their spend *relative to their contract type*, their *data usage patterns*, the *number of customer service interactions*, and even more subtle interactions between these variables.

A fixed rule might state, 'If monthly spend < $30 AND contract is month-to-month, then predict churn.' However, this rule would miss customers with high spend but deteriorating service satisfaction (captured by service calls), or those in a two-year contract approaching its end. Machine learning models can identify non-linear relationships, hidden patterns, and intricate interactions between dozens or hundreds of features that human-defined rules would inevitably overlook or oversimplify, leading to much more accurate and robust predictions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Example: Illustrate the complexity that simple rules miss.
# A simple rule might not capture interactions between features.

# Let's consider a simple rule:
# If 'monthly_spend' < 40 and 'customer_service_calls' > 2, then predict churn.

def simple_churn_rule(row):
    if row['monthly_spend'] < 40 and row['customer_service_calls'] > 2:
        return 1
    return 0

# Apply the rule to our dummy dataframe
df_customers['predicted_churn_rule'] = df_customers.apply(simple_churn_rule, axis=1)

# Compare with actual churn (conceptual)
# In a real scenario, you'd evaluate the rule's performance with metrics.
print("Customers predicted to churn by a simple rule vs. actual churn (first 10 rows):")
display(df_customers[['customer_id', 'monthly_spend', 'customer_service_calls', 'churn_next_quarter', 'predicted_churn_rule']].head(10))

# A more complex ML model would consider all features together, finding more nuanced patterns.
# For example, customers with high data usage but low spend might be at risk, or those with long contract types nearing expiry.

Customers predicted to churn by a simple rule vs. actual churn (first 10 rows):


,customer_id,monthly_spend,customer_service_calls,churn_next_quarter,predicted_churn_rule
0,1,12.49,4,1,1
1,2,22.66,2,0,0
2,3,62.73,4,0,0
3,4,31.87,3,0,1
4,5,59.52,0,0,0
5,6,76.07,4,0,0
6,7,54.23,4,0,0
7,8,61.94,0,0,0
8,9,32.07,3,0,1
9,10,47.76,1,0,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.